# Exp 3 — Image-based mood cue (FER-2013 + MobileNetV2)

**Aim.** Build a lightweight, non-diagnostic **facial mood-cue** signal for the MindSense
app: given a face photo, return a 7-class emotion distribution plus a coarse
`negative-affect / neutral / positive-affect` grouping (prompt decision #2).

**Why image support at all.** Many users chat or text, but a camera check-in can surface
*visible* affect that self-report misses. The cue is strictly supplementary — the app,
metadata, and every output carry *"Screening aid only — not a diagnostic tool."*

**Method (mini-project scope).**
1. FER-2013 (35 887 labelled 48×48 faces) is resized to the app's contract size
   (`config.yaml → image_model.input_size = 224×224`).
2. A **frozen MobileNetV2** (ImageNet weights) runs **forward-only once** to turn every
   face into a 1280-d embedding — backprop through the full backbone on this 8 GB box
   was measured at *minutes per batch*, so the backbone is treated as a fixed feature
   extractor (details in `docs/DECISIONS.md` #22).
3. Four heads are fit on the *same* embeddings and compared with a full
   accuracy / precision / recall / F1 table: logistic regression, random forest,
   XGBoost, and a small MLP (the deep candidate).
4. The best deep candidate is exported — *with the frozen backbone baked into the
   graph* — to `models/face_emotion.onnx`, verified for torch↔onnxruntime parity
   (≤ 1e-4) and the ≤ 20 MB budget, and registered in `models/metadata.json`.

**Reference points.** Human accuracy on FER-2013 is often quoted around **~65%**; the
7 classes are strongly imbalanced (happy ≈ 8 989 vs disgust ≈ 547), so we report
macro-F1 alongside accuracy.

> ⚠️ **Not a diagnostic tool.** This experiment provides a mood *cue* for the app's
> helper screen. It must never be presented as a mental-health assessment.


In [ ]:
# Setup: packages, seed, device, and the real/synthetic banner (Rule 2).
import json
import time
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mindsense import GLOBAL_SEED
from mindsense.models import image as image_mod
from mindsense.utils.io import load_config, repo_path, save_json
from mindsense.utils.plotting import PALETTE, barplot, save_fig, style_axes

PLT = image_mod.GLOBAL_SEED  # alias for clarity in cells below
started = time.perf_counter()
image_mod.set_seed(GLOBAL_SEED)
DEV = image_mod.pick_device("auto")
CFG = image_mod.image_config()
CLASSES = image_mod.emotion_classes()
CUE_GROUPS = image_mod.mood_cue_groups()
print("device:", DEV)
print("input_size:", CFG["input_size"], "| onnx_max_mb:", CFG.get("onnx_max_mb"))
print("classes:", CLASSES)
print("mood_cue_groups:", CUE_GROUPS)

manifest = image_mod.load_manifest()
print("\nmanifest rows:", len(manifest))
print("data_source:", manifest["data_source"].value_counts().to_dict())
print("split:", manifest["split"].value_counts().to_dict())


In [ ]:
# Class balance, split sizes, and a per-class sample grid.
counts = manifest["emotion"].value_counts().reindex(CLASSES)
counts_df = pd.DataFrame({"emotion": CLASSES, "count": counts.astype(int).to_numpy()})
fig = barplot(counts_df, x="emotion", y="count", title="FER-2013 class distribution (real data)")
save_fig(fig, "exp03_01_class_distribution")
print(counts.to_string())
print("most/least frequent ratio: %.1f:1" % (counts.max() / counts.min()))


In [ ]:
# Grid of one example image per emotion (from the test split).
import PIL.Image as PILImage

frame = manifest[manifest["split"] == "test"]
fig, axes = plt.subplots(1, len(CLASSES), figsize=(13, 2.2))
for ax, cls in zip(axes, CLASSES):
    row = frame[frame["emotion"] == cls].iloc[0]
    with PILImage.open(repo_path(row["path"])) as im:
        ax.imshow(im.convert("RGB"))
    ax.set_title(cls, fontsize=10)
    ax.axis("off")
fig.suptitle("One example face per emotion (FER-2013)", fontsize=11)
fig.tight_layout()
save_fig(fig, "exp03_02_sample_grid")


In [ ]:
# Mood-cue grouping (the app-level signal) and its distribution.
cue_by_class = {emotion: image_mod.mood_cue_of(emotion) for emotion in CLASSES}
cue_df = manifest["emotion"].map(cue_by_class).value_counts().reindex(CUE_GROUPS.keys()).reset_index()
cue_df.columns = ["cue", "n"]
print("cue of each class:", cue_by_class)
print(cue_df.to_string(index=False))

fig = barplot(cue_df, x="cue", y="n", title="Mood-cue distribution across FER-2013")
save_fig(fig, "exp03_03_mood_cue")


## Embeddings (one forward-only pass, cached)

`MobileNetV2` features → global average pool → **1280-d vector per face**. The pass is
forward-only (no gradients), so it is cheap even on CPU/MPS, and the result is cached to
`data/processed/fer2013_embeddings/*.npz` so re-running the notebook is instant.

> **Honest scope for a mini project:** the backbone is *not* fine-tuned (impractical on
> this hardware). Performance is therefore bounded by how well generic ImageNet features
> transfer to small emotion faces — a known limitation we'll revisit with real numbers
> below, and exactly why we compare *several* heads instead of a single model.


In [ ]:
# Embed every face once (only the first run computes; later runs hit the cache).
loaders, train_counts, _ = image_mod.build_dataloaders(
    manifest, input_size=tuple(CFG["input_size"]), seed=GLOBAL_SEED
)
backbone = image_mod.FaceEmotionNet(pretrained=True)
frames = image_mod.extract_embeddings(backbone, loaders, DEV, cache=True)
for split, d in frames.items():
    print(f"{split:5s} {d['emb'].shape}  cached={d.get('cached')}")
print("\nembedding wall time so far: %.0fs" % (time.perf_counter() - started))


## Model comparison on shared embeddings

Four heads — logistic regression, random forest, XGBoost, small MLP — are fit on the
*train* embeddings and scored on the held-out **val** (stratified 10% of the train
split) and the official **test** split. All numbers below are real, from this run.


In [ ]:
# Fit + compare all zoo models on the same embeddings; also train the MLP head.
comparison, best_mlp, mlp_history = image_mod.compare_models(
    frames, seed=GLOBAL_SEED, mlp_epochs=20, mlp_batch_size=256
)
rows = []
for name, comp in comparison.items():
    t = comp["test"]
    rows.append({
        "model": name,
        "acc": t["accuracy"], "macro-PR": t["macro_precision"],
        "macro-RC": t["macro_recall"], "macro-F1": t["macro_f1"],
        "wF1": t["weighted_f1"], "fit_s": comp["fit_seconds"],
    })
compare_df = pd.DataFrame(rows).sort_values("macro-F1", ascending=False)
print(compare_df.to_markdown(index=False, numalign="left", stralign="left"))
best_name = compare_df.iloc[0]["model"]
print("\nbest by test macro-F1:", best_name)


In [ ]:
# Comparison figure: accuracy / macro precision / recall / F1 per model.
plot_df = compare_df.melt(
    id_vars=["model"], value_vars=["acc", "macro-PR", "macro-RC", "macro-F1"],
    var_name="metric", value_name="score",
)
fig, ax = plt.subplots(figsize=(9, 4.6))
for i, m in enumerate(plot_df["model"].unique()):
    sub = plot_df[plot_df["model"] == m]
    ax.plot(sub["metric"], sub["score"], marker="o", label=m, color=PALETTE[i % len(PALETTE)])
ax.set_ylim(0, 1)
style_axes(ax, "Model comparison on FER-2013 test set", xlabel="metric", ylabel="score")
ax.legend(frameon=False, fontsize=9)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
save_fig(fig, "exp03_04_model_comparison")

# Per-class recall heatmap for every model.
recall_table = pd.DataFrame({
    name: comp["test"]["per_class_recall"] for name, comp in comparison.items()
})
print("\nper-class recall (test):")
print(recall_table.round(3).to_string())


In [ ]:
# Per-class recall heatmap figure.
fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(recall_table.to_numpy(), cmap="YlGnBu", vmin=0, vmax=1)
ax.set_xticks(range(recall_table.shape[1]), recall_table.columns)
ax.set_yticks(range(recall_table.shape[0]), recall_table.index)
for r in range(recall_table.shape[0]):
    for c in range(recall_table.shape[1]):
        ax.text(c, r, f"{recall_table.iloc[r, c]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Per-class recall by model (FER-2013 test)")
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
save_fig(fig, "exp03_05_per_class_recall")


In [ ]:
# MLP head training history (loss / accuracy per epoch).
if mlp_history:
    hist = pd.DataFrame(mlp_history)
    fig, ax1 = plt.subplots(figsize=(7.5, 4))
    ax1.plot(hist["epoch"], hist["train_loss"], marker="o", color=PALETTE[0], label="train loss")
    ax1.set_ylabel("loss", color=PALETTE[0])
    ax2 = ax1.twinx()
    ax2.plot(hist["epoch"], hist["train_acc"], marker="o", color=PALETTE[1], label="train acc")
    ax2.set_ylabel("accuracy", color=PALETTE[1])
    ax1.set_xlabel("epoch"); ax1.set_title("MLP head training on frozen embeddings")
    fig.tight_layout()
    save_fig(fig, "exp03_06_mlp_training")


## ONNX export + inference-contract verification

The exported graph is the **full pipeline**: ImageNet normalisation (baked-in buffers)
→ MobileNetV2 → MLP head → **raw logits** on a `[0, 1]` RGB `1×3×224×224` input — exactly
what `mindsense.inference.analyze_face` feeds it. We verify torch↔onnxruntime parity
(≤ 1e-4) and the ≤ 20 MB artifact budget, then register provenance in `models/metadata.json`.


In [ ]:
# Rebuild the served net (backbone + best deep head), export, verify parity/size.
net = image_mod.FaceEmotionNet(pretrained=True)
net.classifier = best_mlp
parity_batch = next(iter(loaders["val"]))[0][:8]
onnx_info = image_mod.export_onnx(
    net, repo_path("models", "face_emotion.onnx"),
    input_size=tuple(CFG["input_size"]), parity_batch=parity_batch,
)
assert onnx_info["size_mb"] <= float(CFG.get("onnx_max_mb", 20))
print(json.dumps(onnx_info, indent=2))
print("artifact within budget:", onnx_info["size_mb"] <= float(CFG["onnx_max_mb"]))


In [ ]:
# Optional proof the graph really accepts the app's preprocessing: simulate the
# analyze_face blob path (BGR -> RGB, /255, NCHW) through onnxruntime.
import cv2, onnxruntime as ort
row = manifest[manifest["split"] == "test"].iloc[0]
raw = cv2.imread(str(repo_path(row["path"])))
rgb = cv2.cvtColor(cv2.resize(raw, tuple(CFG["input_size"])), cv2.COLOR_BGR2RGB)
blob = (rgb.astype("float32") / 255.0).transpose(2, 0, 1)[None]
session = ort.InferenceSession(str(repo_path("models", "face_emotion.onnx")), providers=["CPUExecutionProvider"])
logits = session.run(None, {session.get_inputs()[0].name: blob})[0]
probs = np.exp(logits - logits.max(axis=1, keepdims=True)) / np.exp(logits - logits.max(axis=1, keepdims=True)).sum(axis=1, keepdims=True)
top = int(probs[0].argmax())
print("total logits channels:", logits.shape[1])
print("top emotion (model):", CLASSES[top], "| ground truth:", row["emotion"])
print("sum(probs):", probs[0].sum().round(6))


## Grad-CAM: what is the model looking at?

Grad-CAM over the last convolutional block shows the spatial regions driving each
prediction. Backprop through the backbone for a *single-image* CAM is fine on CPU, so
this cell runs on CPU and stays deterministic.


In [ ]:
# Grad-CAM overlaid on six faces (3 correctly and 3 incorrectly predicted).
import matplotlib as mpl

net = net.cpu().eval()
cam_device = image_mod.pick_device("cpu")
sample = manifest[manifest["split"] == "test"].sample(6, random_state=GLOBAL_SEED)
fig, axes = plt.subplots(2, 3, figsize=(10.5, 7))
for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
    with PILImage.open(repo_path(row["path"])) as im:
        img_np = np.asarray(im.convert("RGB")).astype("float32") / 255.0
        batch = torch.from_numpy(img_np.transpose(2, 0, 1)).unsqueeze(0)
        heat = image_mod.gradcam_heatmap(net, batch, cam_device)
    ax.imshow(img_np, alpha=0.55)
    ax.imshow(heat, cmap="jet", alpha=0.45, extent=(0, img_np.shape[1], img_np.shape[0], 0))
    with torch.no_grad():
        pred = int(net(batch).argmax())
    mark = "OK" if pred == image_mod.emotion_classes().index(row["emotion"]) else "miss"
    ax.set_title(f"true={row['emotion']} pred={CLASSES[pred]} ({mark})", fontsize=9)
    ax.axis("off")
fig.suptitle("Grad-CAM for the exported face model (test split)", fontsize=11)
fig.tight_layout()
save_fig(fig, "exp03_07_gradcam")


## Bias, limitations, and how the model is allowed to be used

* **Dataset.** FER-2013 faces are grayscale, centre-cropped, sourced from web image
  search, and biased toward younger, western, bright-casual photos; 48×48 means all fine
  texture is lost when up-sampled to 224. Emotion *labels* are known to be noisy.
* **Backbone frozen (mini-project scope).** No fine-tuning clears the gap to the ~65%
  human benchmark; reported numbers are for the *frozen-embedding* pipeline only.
* **Imbalance.** Happy outnumbers disgust ~16:1; macro-F1 (weighted evenly across all 7
  classes) is the headline metric, and per-class recall shows where the model hides its
  errors.
* **No diagnosis.** The output is a mood *cue*: users may smile in pain, or hide distress
  behind a neutral face. The app always pairs this with self-report risk screening and
  crisis-resource routing (Exp 2 / Exp 6-7), never as a standalone assessment.
* **Privacy.** Faces are processed server-side on this artifact; no images are retained.


## Observations & limitations (real numbers from this run)

*Placeholder — filled from the executed output below.*


In [ ]:
# Final observation data, written to reports/metrics/exp03_image.json.
result = {
    "experiment": "exp03",
    "version": "exp03-v1",
    "device": str(DEV),
    "seed": int(GLOBAL_SEED),
    "data_sources": manifest["data_source"].value_counts().to_dict(),
    "comparison": {
        name: {
            "fit_seconds": comp["fit_seconds"],
            "val": {k: v for k, v in comp["val"].items() if k != "confusion_matrix"},
            "test": {k: v for k, v in comp["test"].items() if k != "confusion_matrix"},
        }
        for name, comp in comparison.items()
    },
    "best_by_test_macro_f1": best_name,
    "onnx": onnx_info,
    "wall_seconds": round(time.perf_counter() - started, 1),
    "generated_at": datetime.now(timezone.utc).isoformat(),
}
save_json(result, repo_path("reports", "metrics", "exp03_image.json"))
print("wrote", repo_path("reports", "metrics", "exp03_image.json"))
print("wall time: %.0fs" % result["wall_seconds"])


## Conclusion

For the mini-project budget this experiment shipped a **working, verified, non-diagnostic
facial mood cue**: a cached MobileNetV2 embedding pass + a four-model comparison on the
same representation, with the best deep candidate exported to ONNX under the app's
inference contract (parity ≤ 1e-4, ≤ 20 MB). The comparison table and per-class recall
figures make the trade-offs visible, and the metadata records *exactly* how the artifact
was built — so anyone can judge how much to trust it.

**How to rerun:** `make data` → `make notebooks` (executes this notebook in place).
